In [1]:
import os
import pandas as pd
import numpy as np
from sodapy import Socrata
from datetime import datetime, timedelta
from google.cloud import bigquery
import urllib.parse

In [2]:
client = Socrata("www.dallasopendata.com", None)

In [3]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "adta5240f22pam-f3b757ff53fa.json"
bq_client = bigquery.Client()

In [4]:
table_id = "adta5240f22pam.adta_5240_project.dallas_arrest_data"

In [5]:
latest_query = f"""
    SELECT MAX(upzdate) as latest_upzdate
    FROM `{table_id}`
"""
latest_result = bq_client.query(latest_query).result()
latest_datetime = [row.latest_upzdate for row in latest_result][0]
f_datetime = latest_datetime.strftime('%Y-%m-%d %H:%M:%S')
f_datetime

'2025-04-29 22:35:27'

In [6]:
print(f"Latest upzdate in BigQuery: {f_datetime}")

Latest upzdate in BigQuery: 2025-04-29 22:35:27


In [7]:
# Define the SQL query with the date injected
query = f"""
    SELECT age,
           ageatarresttime,
           araction,
           aradow,
           ararrestdate,
           ararresttime,
           arbkdate,
           arcurrloc,
           arladdress,
           arlcity,
           arlcounty,
           arldistrict,
           arlzip,
           arpremises,
           arrestyr,
           arstate,
           arweapon,
           birthplace,
           drug,
           drugrelated,
           drugtype,
           employer,
           ethnic,
           eyes,
           hair,
           hcity,
           height,
           hstate,
           hzip,
           nickname,
           occupation,
           race,
           sex,
           tatoo,
           upzdate,
           weight
    WHERE upzdate > '{f_datetime}'
"""

# Encode the query string
safe_string = urllib.parse.quote_plus(query)

# Construct the full URL
url = f'https://www.dallasopendata.com/resource/sdr7-6v3j.csv?$query={safe_string}'


In [8]:
# print the constructed URL to verify it's correct before requesting the data
print(f'Preview URL: {url}')

Preview URL: https://www.dallasopendata.com/resource/sdr7-6v3j.csv?$query=%0A++++SELECT+age%2C%0A+++++++++++ageatarresttime%2C%0A+++++++++++araction%2C%0A+++++++++++aradow%2C%0A+++++++++++ararrestdate%2C%0A+++++++++++ararresttime%2C%0A+++++++++++arbkdate%2C%0A+++++++++++arcurrloc%2C%0A+++++++++++arladdress%2C%0A+++++++++++arlcity%2C%0A+++++++++++arlcounty%2C%0A+++++++++++arldistrict%2C%0A+++++++++++arlzip%2C%0A+++++++++++arpremises%2C%0A+++++++++++arrestyr%2C%0A+++++++++++arstate%2C%0A+++++++++++arweapon%2C%0A+++++++++++birthplace%2C%0A+++++++++++drug%2C%0A+++++++++++drugrelated%2C%0A+++++++++++drugtype%2C%0A+++++++++++employer%2C%0A+++++++++++ethnic%2C%0A+++++++++++eyes%2C%0A+++++++++++hair%2C%0A+++++++++++hcity%2C%0A+++++++++++height%2C%0A+++++++++++hstate%2C%0A+++++++++++hzip%2C%0A+++++++++++nickname%2C%0A+++++++++++occupation%2C%0A+++++++++++race%2C%0A+++++++++++sex%2C%0A+++++++++++tatoo%2C%0A+++++++++++upzdate%2C%0A+++++++++++weight%0A++++WHERE+upzdate+%3E+%272025-04-29+22%3A35%3A

In [12]:
df = pd.read_csv(url)
df = df.where(pd.notnull(df), None)
df.head()

,age,ageatarresttime,araction,aradow,ararrestdate,ararresttime,arbkdate,arcurrloc,arladdress,arlcity,...,height,hstate,hzip,nickname,occupation,race,sex,tatoo,upzdate,weight
0,44.0,44.0,Arrested - Lew Sterrett,Tue,2025-04-29T00:00:00.000,18:00,2025-04-29T22:21:33.000,LS,3410 FORDHAM RD,DALLAS,...,5-10,TX,75216.0,None,None,Black,Female,NaN,2025-04-29 22:52:49,208.0
1,43.0,NaN,Arrested - Lew Sterrett,Tue,2025-04-29T00:00:00.000,19:27,2025-04-29T22:55:04.000,LS,9800 DENTON DR,DALLAS,...,6-01,TX,76207.0,None,None,White,Male,NaN,2025-04-29 23:16:47,150.0
2,44.0,44.0,Arrested - Lew Sterrett,Tue,2025-04-29T00:00:00.000,10:30,2025-04-29T21:42:50.000,LS,3316 TORONTO ST,DALLAS,...,5-0,TX,0.0,None,None,Hispanic or Latino,Male,NaN,2025-04-29 23:27:26,125.0
3,21.0,21.0,Arrested - Lew Sterrett,Tue,2025-04-29T00:00:00.000,10:30,2025-04-29T21:43:34.000,LS,3316 TORONTO ST,DALLAS,...,5-10,TX,0.0,None,None,Hispanic or Latino,Male,NaN,2025-04-29 23:49:02,150.0
4,30.0,30.0,Arrested - Lew Sterrett,Tue,2025-04-29T00:00:00.000,18:20,2025-04-29T23:55:56.000,LS,6464 E NORTHWEST HWY,DALLAS,...,5-05,TX,75243.0,None,None,White,Female,NaN,2025-04-30 00:15:33,140.0


In [13]:
df.shape

(252, 36)

In [14]:
# Clean and convert height (e.g., 5-11 → 5.11)
df['height'] = df['height'].astype(str).str.replace('-', ".", regex=False)
df['height'] = pd.to_numeric(df['height'], errors='coerce')

# Numeric fields (coerce invalids, fill NA if necessary)
df['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).astype(int)

# Boolean fields
for col in ['drugrelated', 'drug', 'nickname']:
    df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})
    df[col] = df[col].fillna(False).astype(bool)

# Datetime fields with coercion
for col in ['upzdate', 'arbkdate', 'ararrestdate']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Columns that should always be treated as string — even if they contain numbers
string_fields = [
    'arldistrict', 'arpremises', 'birthplace', 'drugtype', 'occupation',
    'tatoo', 'sex', 'arcurrloc', 'araction', 'employer', 'aradow', 'hstate',
    'hcity', 'ethnic', 'race', 'eyes', 'hair', 'arweapon', 'arlcounty',
    'arstate', 'arlcity', 'arladdress', 'ararresttime'
]

for col in string_fields:
    df[col] = df[col].astype(str)

# Ensure any remaining object-type columns are stringified
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str)


/var/folders/pm/xyltrhkx3y70nq7rh3h6b5f80000gn/T/ipykernel_93756/1294138332.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
/var/folders/pm/xyltrhkx3y70nq7rh3h6b5f80000gn/T/ipykernel_93756/1294138332.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
/var/folders/pm/xyltrhkx3y70nq7rh3h6b5f80000gn/T/ipykernel_93756/1294138332.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call

In [17]:

# df['height'] = df['height'].str.replace('-', ".")
# df['height'] = df['height'].astype(float)
# df['age'] = df['age'].astype(int)
# df['drugrelated'] = df['drugrelated'].astype(bool)
# df['drug'] = df['drug'].astype(bool)
# df['nickname'] = df['nickname'].astype(bool)
# df['upzdate'] = pd.to_datetime(df['upzdate'])
# df['arbkdate'] = pd.to_datetime(df['arbkdate'])
# df['ararrestdate'] = pd.to_datetime(df['ararrestdate'])
# df['arldistrict'] = df['arldistrict'].astype(str)
# # df['ageatarresttime'] = df['ageatarresttime'].astype(int) ---- Need to be fixed.
# df.head()

In [15]:
df.head()

,age,ageatarresttime,araction,aradow,ararrestdate,ararresttime,arbkdate,arcurrloc,arladdress,arlcity,...,height,hstate,hzip,nickname,occupation,race,sex,tatoo,upzdate,weight
0,44,44.0,Arrested - Lew Sterrett,Tue,2025-04-29,18:00,2025-04-29 22:21:33,LS,3410 FORDHAM RD,DALLAS,...,5.10,TX,75216.0,False,None,Black,Female,nan,2025-04-29 22:52:49,208.0
1,43,NaN,Arrested - Lew Sterrett,Tue,2025-04-29,19:27,2025-04-29 22:55:04,LS,9800 DENTON DR,DALLAS,...,6.01,TX,76207.0,False,None,White,Male,nan,2025-04-29 23:16:47,150.0
2,44,44.0,Arrested - Lew Sterrett,Tue,2025-04-29,10:30,2025-04-29 21:42:50,LS,3316 TORONTO ST,DALLAS,...,5.00,TX,0.0,False,None,Hispanic or Latino,Male,nan,2025-04-29 23:27:26,125.0
3,21,21.0,Arrested - Lew Sterrett,Tue,2025-04-29,10:30,2025-04-29 21:43:34,LS,3316 TORONTO ST,DALLAS,...,5.10,TX,0.0,False,None,Hispanic or Latino,Male,nan,2025-04-29 23:49:02,150.0
4,30,30.0,Arrested - Lew Sterrett,Tue,2025-04-29,18:20,2025-04-29 23:55:56,LS,6464 E NORTHWEST HWY,DALLAS,...,5.05,TX,75243.0,False,None,White,Female,nan,2025-04-30 00:15:33,140.0


In [16]:
job = bq_client.load_table_from_dataframe(df, table_id, job_config=bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"  # Ensures new data is appended
))

print("Data loaded to BigQuery successfully. ", df.shape)

/opt/anaconda3/envs/GCP/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:489: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Data loaded to BigQuery successfully.  (252, 36)
